In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Copy dataset from Drive to local Colab disk (one-time per session)
if not os.path.exists('/content/cleaned_data'):
    !cp -r /content/drive/MyDrive/cleaned_data /content/cleaned_data

dataset_path = '/content/cleaned_data'

# Check if the directory exists
if not os.path.exists(dataset_path):
    print(f"Error: The directory '{dataset_path}' does not exist.")
else:
    print(f"Contents of '{dataset_path}':")
    for item in os.listdir(dataset_path):
        item_path = os.path.join(dataset_path, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")


***Imports and Device initialization***

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt
import numpy as np
import gc
import copy

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

***Transform and Stratified Split***

In [ ]:
norm_stats = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=30, translate=(0.15, 0.15), scale=(0.7, 1.3)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    norm_stats,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    norm_stats
])

# Load data with ImageFolder
full_dataset_train = datasets.ImageFolder(root=dataset_path, transform=train_transform)
full_dataset_val = datasets.ImageFolder(root=dataset_path, transform=val_transform)

# Stratified Split (80/20)
indices = list(range(len(full_dataset_train)))
# ImageFolder automatically sets .targets based on subfolder structure
targets = full_dataset_train.targets

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(full_dataset_train, train_idx)
val_dataset = Subset(full_dataset_val, val_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

class_to_idx = full_dataset_train.class_to_idx
new_classes = full_dataset_train.classes
print(f"Dataset ready with {len(new_classes)} classes.")

***Model Definition***

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer3 and layer4 for fine-tuning (layer3 gets a smaller lr below)
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

# Custom FC layer for 21 classes
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, len(new_classes))
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Discriminative learning rates: deeper backbone layers get smaller lrs
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 5e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4},
], weight_decay=1e-4)

***Train and Validate Functions and Loop***

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.01):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.early_stop = 0, float('inf'), False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter = val_loss, 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader), correct / total

def validate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / len(loader), correct / total

# Clear Memory
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Halve LR when val_loss plateaus for 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

early_stopper = EarlyStopping(patience=3)
best_val_loss = float('inf')
best_state = copy.deepcopy(model.state_dict())
best_epoch = 0

for epoch in range(20):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    v_loss, v_acc = validate(model, val_loader, criterion)
    scheduler.step(v_loss)
    current_lr = optimizer.param_groups[-1]['lr']
    print(f"Epoch {epoch+1} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} | Val Loss: {v_loss:.4f} | LR: {current_lr:.2e}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

    early_stopper(v_loss)
    if early_stopper.early_stop: break

# Restore best weights so downstream evaluation/visualization uses the best model
model.load_state_dict(best_state)
print(f"\nRestored best model from epoch {best_epoch} (val_loss={best_val_loss:.4f})")

In [ ]:
!nvidia-smi

***Visualization***

In [ ]:
def visualize_results(model, loader, num_images=10):
    model.eval()
    inv_map = {v: k for k, v in class_to_idx.items()}
    plt.figure(figsize=(15, 10))
    count = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            for i in range(images.size(0)):
                if count >= num_images: return
                count += 1
                ax = plt.subplot(2, 5, count)
                img = images.cpu().data[i].numpy().transpose((1, 2, 0))
                img = np.clip(np.array([0.229, 0.224, 0.225]) * img + np.array([0.485, 0.456, 0.406]), 0, 1)
                plt.imshow(img)
                color = 'green' if preds[i] == labels[i] else 'red'
                ax.set_title(f"T: {inv_map[labels[i].item()]}\nP: {inv_map[preds[i].item()]}", color=color)
                ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_failures(model, loader, num_images=10):
    model.eval()
    inv_map = {v: k for k, v in class_to_idx.items()}
    plt.figure(figsize=(18, 12))

    failed_count = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            # Find indices where prediction != target
            incorrect_mask = (preds != labels)

            if incorrect_mask.any():
                # Filter current batch for failures
                f_images = images[incorrect_mask]
                f_labels = labels[incorrect_mask]
                f_preds = preds[incorrect_mask]

                for i in range(f_images.size(0)):
                    if failed_count >= num_images:
                        plt.tight_layout()
                        plt.show()
                        return

                    failed_count += 1
                    ax = plt.subplot(num_images // 5 + 1, 5, failed_count)

                    # Un-normalize image
                    img = f_images[i].cpu().numpy().transpose((1, 2, 0))
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = np.clip(std * img + mean, 0, 1)

                    plt.imshow(img)
                    ax.set_title(f"T: {inv_map[f_labels[i].item()]}\nP: {inv_map[f_preds[i].item()]}",
                                 color='red', fontsize=9)
                    ax.axis('off')

    if failed_count == 0:
        print("No failed predictions found in this loader!")
    else:
        plt.tight_layout()
        plt.show()

# Run it on your validation set
visualize_failures(model, val_loader, num_images=10)